# 12 - RoBERTa Dataset Ablation

Bu deney, **Financial PhraseBank'in eğitim havuzuna eklenmesinin gerçekten yarar sağlayıp sağlamadığını**
ölçer.

İki koşul:

1. **Twitter-only**
2. **Twitter + PhraseBank**

Her iki koşulda da aynı `roberta-base`, aynı seed=42, aynı validation ve aynı test seti kullanılır.

Sabit ayarlar:
- epoch = 4
- learning rate = 2e-5
- train batch = 16
- eval batch = 32
- max length = 128
- weight decay = 0.01
- class weights = açık
- model seçimi = validation Macro-F1

Notebook internal test sonuçlarını üretir. Ayrıca proje içindeki S&P 500 ve Reuters anotasyon dosyaları bulunursa
iki modeli bu dış testlerde de otomatik değerlendirir.


<!-- thesis-review-note -->
## Çalışma Notu

- Amaç: RoBERTa icin Twitter-only ile Twitter + PhraseBank egitim kosullarini karsilastirir.
- Girdi: Ayni test/validation duzeninde iki egitim veri kosulu.
- Çıktı ve değerlendirme: PhraseBank eklemenin Macro-F1 ve ozellikle neutral sinifina etkisini olcer.
- Sıra notu: Notebook numarasi deney akışındaki yerini gösterir; önceki numaralar tamamlanmadan kalıcı sonuç yorumları güncellenmemelidir.
- Sonuç güvenliği: Bu dosyadaki mevcut output hücreleri ve kalıcı sonuç dosyaları korunur.


In [1]:

from thesis_utils import PROJECT_ROOT, paths

from pathlib import Path
import os, re, gc, json, random, inspect, warnings
import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "roberta-base"
RANDOM_STATE = 42
MAX_LENGTH = 128
EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 2
FORCE_RETRAIN = False

TEXT_COL = "input_text"
LABEL_COL = "label"
LABEL_ID_COL = "label_id"

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABELS_ORDER = ["negative", "neutral", "positive"]
NUM_LABELS = 3

SPLIT_DIR = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
TRAIN_PATH = SPLIT_DIR / "train_df.parquet"
VAL_PATH = SPLIT_DIR / "val_df.parquet"
TEST_PATH = SPLIT_DIR / "test_df.parquet"

CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
ABLATION_ROOT = CHECKPOINT_ROOT / "dataset_ablation_roberta"
ABLATION_RESULTS_ROOT = paths.MODEL_RESULTS_ROOT / "dataset_ablation_roberta"
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)
ABLATION_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Ablation checkpoint root:", ABLATION_ROOT)
print("Ablation results root   :", ABLATION_RESULTS_ROOT)


Device: cpu
Ablation root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\dataset_ablation_roberta


In [2]:

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Split bulunamadı: {p}\n"
            "Önce 04_train_plain_sentiment_models.ipynb ile splitleri oluştur."
        )

combined_train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

for name, df in [("train", combined_train_df), ("val", val_df), ("test", test_df)]:
    for col in [TEXT_COL, LABEL_COL, LABEL_ID_COL]:
        if col not in df.columns:
            raise ValueError(f"{name}: eksik kolon -> {col}")

if "source_dataset" not in combined_train_df.columns:
    raise ValueError("train_df içinde source_dataset yok.")

combined_train_df["source_dataset"] = combined_train_df["source_dataset"].astype(str).str.strip()

twitter_train_df = combined_train_df[
    combined_train_df["source_dataset"] == "TwitterFinancialNewsSentiment"
].copy().reset_index(drop=True)

phrasebank_train_df = combined_train_df[
    combined_train_df["source_dataset"] == "FinancialPhraseBank"
].copy().reset_index(drop=True)

if len(twitter_train_df) == 0:
    raise ValueError("TwitterFinancialNewsSentiment train örneği bulunamadı.")
if len(phrasebank_train_df) == 0:
    raise ValueError("FinancialPhraseBank train örneği bulunamadı.")

print("Twitter-only train:", twitter_train_df.shape)
print(twitter_train_df[LABEL_COL].value_counts())
print("\nPhraseBank portion:", phrasebank_train_df.shape)
print(phrasebank_train_df[LABEL_COL].value_counts())
print("\nCombined train:", combined_train_df.shape)
print("\nValidation:", val_df.shape)
print("Test:", test_df.shape)


Twitter-only train: (8110, 24)
label
neutral     5264
positive    1630
negative    1216
Name: count, dtype: int64

PhraseBank portion: (4840, 24)
label
neutral     2873
positive    1363
negative     604
Name: count, dtype: int64

Combined train: (12950, 24)

Validation: (1432, 24)
Test: (2386, 24)


In [3]:

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def prepare_hf_dataset(df, tokenizer, text_col=TEXT_COL, label_id_col=LABEL_ID_COL):
    temp = df[[text_col, label_id_col]].copy()
    temp[text_col] = temp[text_col].astype(str)
    temp[label_id_col] = temp[label_id_col].astype(int)
    temp = temp.rename(columns={text_col: "text", label_id_col: "labels"})

    ds = Dataset.from_pandas(temp.reset_index(drop=True))

    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    ds = ds.map(tok, batched=True)
    keep = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in ds.column_names:
        keep.append("token_type_ids")
    ds.set_format(type="torch", columns=keep)
    return ds

def prepare_eval_dataset(df, tokenizer, text_col, gold_col):
    temp = df[[text_col, gold_col]].copy()
    temp[text_col] = temp[text_col].astype(str)
    temp[gold_col] = temp[gold_col].astype(str).str.lower().str.strip()
    temp = temp[temp[gold_col].isin(LABEL2ID)].copy()
    temp["labels"] = temp[gold_col].map(LABEL2ID).astype(int)
    temp = temp.rename(columns={text_col: "text"})

    ds = Dataset.from_pandas(temp[["text", "labels"]].reset_index(drop=True))

    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    ds = ds.map(tok, batched=True)
    keep = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in ds.column_names:
        keep.append("token_type_ids")
    ds.set_format(type="torch", columns=keep)
    return temp.reset_index(drop=True), ds

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

def calculate_class_weights(df):
    y = df[LABEL_ID_COL].astype(int).to_numpy()
    counts = np.bincount(y, minlength=NUM_LABELS)
    weights = [len(y) / (NUM_LABELS * c) if c else 1.0 for c in counts]
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = self.class_weights_tensor.to(logits.device) if self.class_weights_tensor is not None else None
        loss = nn.CrossEntropyLoss(weight=weights)(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

def trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    if "processing_class" in sig.parameters:
        return {"processing_class": tokenizer}
    if "tokenizer" in sig.parameters:
        return {"tokenizer": tokenizer}
    return {}

def build_training_args(output_dir):
    sig = inspect.signature(TrainingArguments.__init__)
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
    )
    if "data_seed" in sig.parameters:
        kwargs["data_seed"] = RANDOM_STATE
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "epoch"
    if "save_strategy" in sig.parameters:
        kwargs["save_strategy"] = "epoch"
    if "save_total_limit" in sig.parameters:
        kwargs["save_total_limit"] = SAVE_TOTAL_LIMIT
    if "load_best_model_at_end" in sig.parameters:
        kwargs["load_best_model_at_end"] = True
    if "metric_for_best_model" in sig.parameters:
        kwargs["metric_for_best_model"] = "f1_macro"
    if "greater_is_better" in sig.parameters:
        kwargs["greater_is_better"] = True
    if "fp16" in sig.parameters:
        kwargs["fp16"] = torch.cuda.is_available()
    return TrainingArguments(**kwargs)

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


In [4]:

EXPERIMENTS = [
    {
        "experiment": "twitter_only",
        "display_name": "RoBERTa - Twitter only",
        "train_df": twitter_train_df,
    },
    {
        "experiment": "twitter_plus_phrasebank",
        "display_name": "RoBERTa - Twitter + PhraseBank",
        "train_df": combined_train_df,
    },
]

trained_runs = {}

for exp in EXPERIMENTS:
    exp_name = exp["experiment"]
    train_part = exp["train_df"]

    run_dir = ABLATION_ROOT / exp_name
    checkpoint_dir = run_dir / "checkpoints"
    final_model_dir = run_dir / "final_model"
    results_dir = ABLATION_RESULTS_ROOT / exp_name / "results"

    for p in [checkpoint_dir, final_model_dir, results_dir]:
        p.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print(exp["display_name"])
    print("#" * 100)
    print("Train N:", len(train_part))

    set_all_seeds(RANDOM_STATE)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    train_ds = prepare_hf_dataset(train_part, tokenizer)
    val_ds = prepare_hf_dataset(val_df, tokenizer)
    test_ds = prepare_hf_dataset(test_df, tokenizer)

    model_exists = (
        (final_model_dir / "config.json").exists()
        and (
            any(final_model_dir.glob("model*.safetensors"))
            or (final_model_dir / "pytorch_model.bin").exists()
        )
    )

    if model_exists and not FORCE_RETRAIN:
        print("Final model bulundu -> eğitim atlanıyor.")
        model = AutoModelForSequenceClassification.from_pretrained(final_model_dir)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        )

    model.to(DEVICE)

    trainer = WeightedTrainer(
        model=model,
        args=build_training_args(checkpoint_dir),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights_tensor=calculate_class_weights(train_part),
        **trainer_tokenizer_kwargs(tokenizer),
    )

    if not model_exists or FORCE_RETRAIN:
        trainer.train()
        trainer.save_model(str(final_model_dir))
        tokenizer.save_pretrained(str(final_model_dir))

    trained_runs[exp_name] = {
        "display_name": exp["display_name"],
        "trainer": trainer,
        "tokenizer": tokenizer,
        "test_ds": test_ds,
        "results_dir": results_dir,
        "final_model_dir": final_model_dir,
        "n_train": len(train_part),
    }

print("\nİki ablation modeli hazır.")



####################################################################################################
RoBERTa - Twitter only
####################################################################################################
Train N: 8110


Map:   0%|          | 0/8110 [00:00<?, ? examples/s]

Map:   0%|          | 0/1432 [00:00<?, ? examples/s]

Map:   0%|          | 0/2386 [00:00<?, ? examples/s]

Final model bulundu -> eğitim atlanıyor.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


İki ablation modeli hazır.


In [5]:

def evaluate_predictions(trainer, dataset, source_df, experiment, dataset_name, results_dir):
    output = trainer.predict(dataset)
    logits = output.predictions
    y_true = output.label_ids.astype(int)
    probs = softmax_np(logits)
    y_pred = probs.argmax(axis=1).astype(int)

    p_macro, r_macro, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, class_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0,1,2], average=None, zero_division=0
    )

    metrics = {
        "experiment": experiment,
        "dataset": dataset_name,
        "n_eval": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "f1_negative": class_f1[0],
        "f1_neutral": class_f1[1],
        "f1_positive": class_f1[2],
    }

    pred_df = source_df.reset_index(drop=True).copy()
    pred_df["gold_label"] = [ID2LABEL[int(x)] for x in y_true]
    pred_df["prediction"] = [ID2LABEL[int(x)] for x in y_pred]
    pred_df["correct"] = y_true == y_pred
    pred_df["confidence"] = probs.max(axis=1)
    pred_df["prob_negative"] = probs[:,0]
    pred_df["prob_neutral"] = probs[:,1]
    pred_df["prob_positive"] = probs[:,2]

    pred_df.to_csv(
        results_dir / f"{dataset_name}_predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )
    return metrics

all_results = []

for exp_name, run in trained_runs.items():
    m = evaluate_predictions(
        run["trainer"],
        run["test_ds"],
        test_df,
        exp_name,
        "internal_test",
        run["results_dir"],
    )
    m["n_train"] = run["n_train"]
    all_results.append(m)

internal_results_df = pd.DataFrame(all_results)
display(internal_results_df.round(6))


,experiment,dataset,n_eval,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive,n_train
0,twitter_only,internal_test,2386,0.903604,0.866182,0.898080,0.881030,0.904657,0.845333,0.929680,0.868077,8110
1,twitter_plus_phrasebank,internal_test,2386,0.894803,0.857787,0.881298,0.867862,0.895845,0.821705,0.925041,0.856840,12950


In [6]:

def read_sp500_external():
    annotation_dir = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
    if not annotation_dir.exists():
        print("S&P 500 klasörü yok -> atlanacak.")
        return None

    files = sorted([
        p for p in annotation_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".xlsx", ".csv"}
        and not p.name.startswith("~$")
    ])

    parts = []
    for p in files:
        try:
            temp = pd.read_excel(p) if p.suffix.lower() == ".xlsx" else pd.read_csv(p, encoding="utf-8-sig")
            temp.columns = [str(c).strip() for c in temp.columns]
            parts.append(temp)
        except Exception:
            pass

    if not parts:
        return None

    df = pd.concat(parts, ignore_index=True)

    if "text_en" not in df.columns:
        raise ValueError("S&P 500 batchlerinde text_en bulunamadı.")

    df["gold_label"] = df["final_label"] if "final_label" in df.columns else pd.NA

    if "chatgpt_label" in df.columns:
        df["gold_label"] = df["gold_label"].fillna(df["chatgpt_label"])

    df["gold_label"] = df["gold_label"].astype(str).str.lower().str.strip()
    df["text_en"] = df["text_en"].astype(str).str.strip()

    df = df[
        df["gold_label"].isin(LABEL2ID)
        & (df["text_en"] != "")
        & (df["text_en"].str.lower() != "nan")
    ].copy()

    if "annotation_id" in df.columns:
        df = df.drop_duplicates(subset=["annotation_id"], keep="first")

    return df.reset_index(drop=True)

sp500_df = read_sp500_external()

if sp500_df is not None:
    print("S&P 500 N:", len(sp500_df))
    print(sp500_df["gold_label"].value_counts())


S&P 500 N: 1030
gold_label
positive    384
neutral     332
negative    314
Name: count, dtype: int64


In [7]:

def normalize_col(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    return c.replace("\n", "_").replace("\r", "_")

def unique_cols(cols):
    seen, out = {}, []
    for c in cols:
        c = normalize_col(c)
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")
    return out

def read_reuters_batch(path):
    if path.suffix.lower() == ".xlsx":
        raw = pd.read_excel(path, header=None, dtype=object)
    else:
        raw = pd.read_csv(path, header=None, encoding="utf-8-sig", dtype=object)

    raw = raw.dropna(how="all")
    header_idx = None

    for idx in range(len(raw)):
        vals = raw.iloc[idx].astype(str).str.strip().str.lower().tolist()
        if "annotation_id" in vals:
            header_idx = idx
            break

    if header_idx is not None:
        header = raw.iloc[header_idx].tolist()
        df = raw.iloc[header_idx+1:].copy()
        df.columns = unique_cols(header)
    else:
        df = pd.read_excel(path) if path.suffix.lower() == ".xlsx" else pd.read_csv(path, encoding="utf-8-sig")
        df.columns = unique_cols(df.columns)

    df = df.dropna(how="all").copy()

    if "text_en" not in df.columns:
        df["text_en"] = pd.NA

    for fallback in ["text", "source_text", "headline", "title"]:
        if fallback in df.columns:
            df["text_en"] = df["text_en"].fillna(df[fallback])

    if "final_label" not in df.columns:
        df["final_label"] = pd.NA
    if "chatgpt_label" not in df.columns:
        df["chatgpt_label"] = pd.NA

    return df

def read_reuters_external():
    annotation_dir = paths.REUTERS_ANNOTATION_DIR
    if not annotation_dir.exists():
        print("Reuters klasörü yok -> atlanacak.")
        return None

    files = sorted([
        p for p in annotation_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".xlsx", ".csv"}
        and "annotation" in p.name.lower()
        and "batch" in p.name.lower()
        and not p.name.startswith("~$")
    ])

    parts = []
    for p in files:
        try:
            parts.append(read_reuters_batch(p))
        except Exception as exc:
            print("Atlanan Reuters dosyası:", p.name, "|", exc)

    if not parts:
        return None

    df = pd.concat(parts, ignore_index=True)
    df["gold_label"] = df["final_label"].fillna(df["chatgpt_label"])
    df["gold_label"] = df["gold_label"].astype(str).str.lower().str.strip()
    df["text_en"] = df["text_en"].astype(str).str.strip()

    df = df[
        df["gold_label"].isin(LABEL2ID)
        & (df["text_en"] != "")
        & (df["text_en"].str.lower() != "nan")
    ].copy()

    if "annotation_id" in df.columns:
        df = df.drop_duplicates(subset=["annotation_id"], keep="first")

    return df.reset_index(drop=True)

reuters_df = read_reuters_external()

if reuters_df is not None:
    print("Reuters N:", len(reuters_df))
    print(reuters_df["gold_label"].value_counts())


Reuters N: 5000
gold_label
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64


In [8]:

for exp_name, run in trained_runs.items():
    trainer = run["trainer"]
    tokenizer = run["tokenizer"]

    if sp500_df is not None:
        sp500_clean, sp500_ds = prepare_eval_dataset(
            sp500_df, tokenizer, "text_en", "gold_label"
        )
        m = evaluate_predictions(
            trainer, sp500_ds, sp500_clean,
            exp_name, "sp500_external", run["results_dir"]
        )
        m["n_train"] = run["n_train"]
        all_results.append(m)

    if reuters_df is not None:
        reuters_clean, reuters_ds = prepare_eval_dataset(
            reuters_df, tokenizer, "text_en", "gold_label"
        )
        m = evaluate_predictions(
            trainer, reuters_ds, reuters_clean,
            exp_name, "reuters_external", run["results_dir"]
        )
        m["n_train"] = run["n_train"]
        all_results.append(m)

ablation_results_df = pd.DataFrame(all_results)

result_path = ABLATION_RESULTS_ROOT / "roberta_dataset_ablation_results.csv"
ablation_results_df.to_csv(result_path, index=False, encoding="utf-8-sig")

display(
    ablation_results_df[
        [
            "experiment", "dataset", "n_train", "n_eval",
            "accuracy", "f1_macro", "f1_weighted",
            "f1_negative", "f1_neutral", "f1_positive"
        ]
    ].round(6)
)

print("Saved:", result_path)


Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

,experiment,dataset,n_train,n_eval,accuracy,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive
0,twitter_only,internal_test,8110,2386,0.903604,0.881030,0.904657,0.845333,0.929680,0.868077
1,twitter_plus_phrasebank,internal_test,12950,2386,0.894803,0.867862,0.895845,0.821705,0.925041,0.856840
2,twitter_only,sp500_external,8110,1030,0.798058,0.798912,0.800339,0.804196,0.763540,0.829001
3,twitter_only,reuters_external,8110,5000,0.635600,0.646471,0.656632,0.736481,0.545980,0.656951
4,twitter_plus_phrasebank,sp500_external,12950,1030,0.795146,0.796781,0.797249,0.816189,0.761394,0.812760
5,twitter_plus_phrasebank,reuters_external,12950,5000,0.617800,0.630508,0.638934,0.730653,0.531458,0.629415


Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\dataset_ablation_roberta\roberta_dataset_ablation_results.csv


In [9]:

difference_rows = []

for dataset_name in sorted(ablation_results_df["dataset"].unique()):
    subset = ablation_results_df[
        ablation_results_df["dataset"] == dataset_name
    ].set_index("experiment")

    if not {"twitter_only", "twitter_plus_phrasebank"}.issubset(subset.index):
        continue

    a = subset.loc["twitter_only"]
    b = subset.loc["twitter_plus_phrasebank"]

    difference_rows.append({
        "dataset": dataset_name,
        "accuracy_delta_phrasebank": b["accuracy"] - a["accuracy"],
        "macro_f1_delta_phrasebank": b["f1_macro"] - a["f1_macro"],
        "weighted_f1_delta_phrasebank": b["f1_weighted"] - a["f1_weighted"],
        "negative_f1_delta_phrasebank": b["f1_negative"] - a["f1_negative"],
        "neutral_f1_delta_phrasebank": b["f1_neutral"] - a["f1_neutral"],
        "positive_f1_delta_phrasebank": b["f1_positive"] - a["f1_positive"],
    })

difference_df = pd.DataFrame(difference_rows)

difference_path = ABLATION_RESULTS_ROOT / "roberta_dataset_ablation_differences.csv"
difference_df.to_csv(difference_path, index=False, encoding="utf-8-sig")

print("PhraseBank katkı farkları:")
display(difference_df.round(6))

print("\nYorum:")
print("- delta > 0 : PhraseBank eklemek metriği artırdı.")
print("- delta < 0 : Twitter-only daha iyi.")
print("- İşaret dataset'e göre değişirse domain shift / label-semantics tartışması güçlenir.")

print("\nSaved:", difference_path)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


PhraseBank katkı farkları:


,dataset,accuracy_delta_phrasebank,macro_f1_delta_phrasebank,weighted_f1_delta_phrasebank,negative_f1_delta_phrasebank,neutral_f1_delta_phrasebank,positive_f1_delta_phrasebank
0,internal_test,-0.008801,-0.013168,-0.008813,-0.023628,-0.004639,-0.011237
1,reuters_external,-0.017800,-0.015962,-0.017698,-0.005829,-0.014522,-0.027536
2,sp500_external,-0.002913,-0.002131,-0.003091,0.011993,-0.002146,-0.016241



Yorum:
- delta > 0 : PhraseBank eklemek metriği artırdı.
- delta < 0 : Twitter-only daha iyi.
- İşaret dataset'e göre değişirse domain shift / label-semantics tartışması güçlenir.

Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\dataset_ablation_roberta\roberta_dataset_ablation_differences.csv


## Bana göndermen gerekenler

Notebook bitince şu iki dosyayı yükle:

- `outputs/finetuned_model_results/dataset_ablation_roberta/roberta_dataset_ablation_results.csv`
- `outputs/finetuned_model_results/dataset_ablation_roberta/roberta_dataset_ablation_differences.csv`

Özellikle `macro_f1_delta_phrasebank` ve `neutral_f1_delta_phrasebank` değerlerine bakacağız.
